# Drift Monitoring — Inference Monitoring Notebook

> 📊 **Note**: this notebook contains interactive output cells (Evidently reports, plotly charts). They render fully in JupyterLab; static viewers like the GitHub/GitLab renderer strip the JavaScript.

End-to-end drift monitoring for the deployed **bank-marketing** endpoint, run
entirely inside this notebook. All infrastructure (the `bank_marketing`
Athena tables, the endpoint's inference-capture path, the MLflow app) is
provisioned by the workshop CloudFormation stack (`scripts/deploy-workshop.sh`).
This notebook **creates no infrastructure** — no Lambda, SQS, EventBridge
schedule, SNS topic, or CloudWatch alarm. It computes drift and writes results
directly to Athena.

You focus on the parts that need a data scientist:

1. **Setup** — load config, confirm the monitoring tables exist
2. **Generate drifted data** *(dev/test only)* — fabricate traffic that should trip the detector
3. **Send predictions** — invoke the endpoint so there's something to monitor
4. **Apply ground truth** — simulate labels for model-drift metrics
5. **Detect drift** — the core: data drift + model drift with Evidently, written to Athena and logged to MLflow

> **Rule**: never send predictions (Section 3) *after* running the ground-truth simulator (Section 4) — those rows would have NULL ground_truth and skew model-drift metrics. The cell order enforces this.


## Prerequisites

This notebook assumes the following already exist (all created by
`scripts/deploy-workshop.sh` plus the earlier labs). If any are missing,
create them before continuing:

| Prerequisite | How it's created |
|---|---|
| SageMaker domain, S3 data bucket, MLflow app, `bank_marketing` Athena tables (`training_data`, `evaluation_data`, `inference_responses`, `monitoring_responses`, `ground_truth_updates`), inference-capture path | Workshop stack (`scripts/deploy-workshop.sh` → `templates/`) |
| A trained + **Approved** model in the `bank-prediction-XGBoostModel` model package group | Lab 3A / 3C |
| A live `bank-marketing-<timestamp>` endpoint logging to `inference_responses` | Lab 3A |

The `.env` file (populated by the domain lifecycle script) carries `DATA_S3_BUCKET`
and `AWS_DEFAULT_REGION`; everything else resolves from the workshop stack
outputs via `src/config/config.py`. The endpoint is discovered live (the most
recent InService `bank-marketing-*` endpoint) — nothing to hand-edit.


## 1. Setup

In [ ]:
# Load the shared environment written once by lab0-setup/setup.ipynb.
# No per-lab discovery or hardcoding — lab0-setup is the single source of truth.
import os
from pathlib import Path
from dotenv import load_dotenv

_p = Path.cwd()
for _cand in [_p, *_p.parents]:
    if (_cand / '.env').exists():
        load_dotenv(_cand / '.env', override=False)
        _ENV_PATH = _cand / '.env'
        break
else:
    raise RuntimeError('No .env found. Run lab0-setup/setup.ipynb first.')

PROJECT_NAME = os.environ.get('PROJECT_NAME', 'bank-marketing-prediction')
print(f'\u2713 Loaded shared environment from {_ENV_PATH}')
print(f'  PROJECT_NAME={PROJECT_NAME}')

# --- lab5-only: install this monitoring package so `import src` resolves. ---
# Only lab5d/lab5f need it, so it lives here rather than in the shared
# lab0-setup. (Temporary: pending the planned rename of src/ to a real package
# name, after which this becomes an ordinary dependency.)
import sys, subprocess
# lab5 notebooks run from lab5-monitoring/; find the dir that holds pyproject.toml.
_lab5 = next((p for p in [Path.cwd(), *Path.cwd().parents]
              if (p / 'pyproject.toml').exists() and (p / 'src').is_dir()), Path.cwd())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(_lab5), '--quiet'],
               check=False)
print(f'\u2713 lab5 monitoring package installed from {_lab5}')


In [ ]:
import sys, os, json, time, logging, uuid, importlib
from pathlib import Path
from datetime import datetime, timedelta

import boto3
import pandas as pd
import numpy as np

# The shared repo-root .env was written by lab0-setup/setup.ipynb.
# lab5/ (this dir) is still put on sys.path so src/ imports work.
project_root = Path.cwd()
_env_dir = next((p for p in [project_root, *project_root.parents]
                 if (p / '.env').exists()), None)
if _env_dir is None:
    raise RuntimeError('No .env found. Run lab0-setup/setup.ipynb first.')

# Force lab5/ onto sys.path so src/ is importable.
# Also remove any stale 'src' entries from a broken previous install.
_lab5_str = str(project_root)
if _lab5_str not in sys.path:
    sys.path.insert(0, _lab5_str)
for key in [k for k in list(sys.modules.keys()) if k == 'src' or k.startswith('src.')]:
    del sys.modules[key]

from dotenv import load_dotenv
load_dotenv(_env_dir / '.env', override=True)

from src.config.config import (
    ATHENA_DATABASE, DATA_S3_BUCKET, AWS_DEFAULT_REGION,
    ATHENA_OUTPUT_S3,
    ATHENA_TRAINING_TABLE, ATHENA_EVALUATION_TABLE,
    ATHENA_INFERENCE_TABLE,
    ATHENA_GROUND_TRUTH_UPDATES_TABLE,
    CSV_DRIFTED_DATA,
    SAGEMAKER_EXEC_ROLE, MLFLOW_MODEL_NAME,
    MONITORING_DATA_DRIFT_LOOKBACK_DAYS,
    GROUND_TRUTH_SIM_ACCURACY,
    GROUND_TRUTH_SIM_FEATURE_DRIFT_IMPACT,
    GROUND_TRUTH_SIM_MODEL_DRIFT_MAG,
    ENDPOINT_NAME as _CFG_ENDPOINT_NAME,
    PREDICTION_COLUMN,
)
from src.config.config import resolve_endpoint_name
from src.config import schema
from src.train_pipeline.athena.athena_client import AthenaClient

TRAINING_FEATURES = schema.feature_names()
TARGET_COLUMN = schema.target_column()

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Resolve the live endpoint created by Lab 3A (pattern bank-marketing-<timestamp>).
# resolve_endpoint_name() returns the pinned ENDPOINT_NAME if set, else the most
# recent InService endpoint whose name starts with the bank-marketing prefix.
ENDPOINT_NAME = resolve_endpoint_name()
if not ENDPOINT_NAME:
    raise RuntimeError(
        'No InService bank-marketing endpoint found. Deploy it from '
        'lab3-model-build/lab-3a_traditional_ml_experimenation.ipynb first, '
        'or pin endpoint.name in src/config/config.yaml.'
    )
REGION = AWS_DEFAULT_REGION

runtime_client = boto3.client('sagemaker-runtime', region_name=REGION)
sm_client = boto3.client('sagemaker', region_name=REGION)
athena_client = AthenaClient()

print(f'Region: {REGION}')
print(f'Endpoint: {ENDPOINT_NAME}')
print(f'Athena DB: {ATHENA_DATABASE}')
print(f'Target column: {TARGET_COLUMN}')
print(f'Features: {len(TRAINING_FEATURES)}')
print(f'S3 Bucket: {DATA_S3_BUCKET}')
print(f'MLflow URI: {os.getenv("MLFLOW_TRACKING_URI", "NOT SET")}')
print(f'Data Drift Lookback: {MONITORING_DATA_DRIFT_LOOKBACK_DAYS} days')

### 1.1 Monitoring tables (provisioned by the workshop stack)

The Athena tables this notebook reads and writes — `inference_responses`,
`monitoring_responses`, and `ground_truth_updates` — are all created by the
**workshop CloudFormation stack** (`deploy-workshop.sh` →
`templates/4-inference-capture.yaml`), in the `bank_marketing` database.

This notebook **creates no infrastructure**: it only writes rows into those
tables. There is no drift-monitoring Lambda, SQS queue, EventBridge schedule,
SNS topic, or CloudWatch alarm — drift is computed and persisted inline below.


In [ ]:
# Read-only: confirm the monitoring tables exist (created by the workshop
# stack). Creates nothing. If a table is missing, (re)deploy the workshop
# stack with scripts/deploy-workshop.sh — do NOT create tables from here.
from src.config.config import (
    ATHENA_DATABASE, ATHENA_INFERENCE_TABLE,
    ATHENA_MONITORING_RESPONSES_TABLE, ATHENA_GROUND_TRUTH_UPDATES_TABLE,
)

MONITORING_TABLE_NAME = ATHENA_MONITORING_RESPONSES_TABLE

_required_tables = [
    ATHENA_INFERENCE_TABLE,
    ATHENA_MONITORING_RESPONSES_TABLE,
    ATHENA_GROUND_TRUTH_UPDATES_TABLE,
]

print('━' * 72)
print(f'MONITORING TABLES (in {ATHENA_DATABASE}, created by the workshop stack)')
print('━' * 72)
_glue = boto3.client('glue', region_name=AWS_DEFAULT_REGION)
_missing = []
for _t in _required_tables:
    try:
        _glue.get_table(DatabaseName=ATHENA_DATABASE, Name=_t)
        print(f'  ✓ {ATHENA_DATABASE}.{_t}')
    except Exception:
        _missing.append(_t)
        print(f'  ✗ {ATHENA_DATABASE}.{_t}  (MISSING)')
print('━' * 72)
if _missing:
    print('Missing table(s):', ', '.join(_missing))
    print('Redeploy the workshop stack (scripts/deploy-workshop.sh) — the\n'
          '4-inference-capture.yaml table-creator provisions all three.\n'
          'This notebook never creates tables itself.')


### 1.2 What this notebook does

Sends predictions to the live endpoint, applies simulated ground truth,
computes **data drift** and **model drift** with Evidently, writes the run
to `monitoring_responses`, and logs the Evidently reports + metrics to
**MLflow**. The `monitoring_responses` / `inference_responses` /
`ground_truth_updates` tables then back the **QuickSight** governance
dashboard (Lab 5e).


In [ ]:
# No drift-monitoring infrastructure to confirm — this notebook uses only
# the workshop-provisioned Athena tables (checked in Section 1.1) plus the
# MLflow app. Drift results are written directly to Athena in Section 5.4.
print(f'Endpoint            : {ENDPOINT_NAME}')
print(f'Athena database     : {ATHENA_DATABASE}')
print(f'Monitoring table    : {ATHENA_DATABASE}.{MONITORING_TABLE_NAME}')
print(f'MLflow tracking URI : {os.getenv("MLFLOW_TRACKING_URI", "NOT SET")}')


## 2. Generate Drifted Test Data *(dev/test only)*

Fabricate a drifted CSV the detector should flag, so Section 5 has something to
find. In production you'd skip this entirely — real traffic supplies the drift.
Drift parameters live in `src/config/config.yaml` → `drift_generation.default_drift`.

In [ ]:
# Generate drifted dataset (configured via config.yaml)
import subprocess
import sys

print('=' * 80)
print('GENERATING DRIFTED TEST DATA')
print('=' * 80)
print('\nThis will create data/drifted_data.csv with configurable drift amounts.')
print('Drift parameters are read from: src/config/config.yaml → drift_generation.default_drift\n')

result = subprocess.run(
    [sys.executable, 'src/drift_monitoring/generate_drift_dataset.py'],
    cwd=str(project_root),
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode != 0:
    print('❌ Error generating drifted dataset:')
    print(result.stderr)
else:
    print('\n✅ Drifted dataset generated successfully!')
    print('   File: data/drifted_data.csv')
    print('   To use this data, send it to your endpoint for inference testing.')

## 3. Send Predictions to the Endpoint

Every cell here invokes the deployed endpoint. The custom handler auto-logs each
prediction to Athena via SQS → Lambda → `inference_responses` (batches of 10 or
every 30s). Most invocations send intentionally-drifted samples from Section 2;
a small sanity check confirms the endpoint is alive.

### 3.1 Helper: JSON invocation function

In [ ]:
def invoke_endpoint_json(endpoint_name, feature_dict):
    """Invoke endpoint with JSON format (custom handler with Athena logging).

    Dataset-agnostic: accepts whichever key the handler uses for the positive
    class ("yes" for Lab 3A, or "positive"/"fraud" for other handlers).
    """
    import json
    import time

    payload = json.dumps(feature_dict)
    start = time.time()

    response = runtime_client.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='application/json',
        Body=payload
    )

    latency_ms = (time.time() - start) * 1000
    result = json.loads(response['Body'].read().decode())

    # Lab 3A's handler returns:
    #   {"predictions": [0/1], "probabilities": {"yes": [...], "no": [...]}}
    prediction = result["predictions"][0]
    probs = result.get("probabilities", {})
    # Other handlers label the positive class "positive" or (legacy) "fraud",
    # so accept any of them and fail loudly rather than silently yielding None.
    for _key in ("yes", "positive", "fraud"):
        if probs.get(_key):
            prob_positive = probs[_key][0]
            break
    else:
        raise KeyError(
            f"No positive-class probability in handler response: {sorted(probs)}"
        )

    return {'prediction': prediction, 'probability_positive': prob_positive}, latency_ms


### 3.2 Sanity check: one hard-coded payload

In [ ]:
# Single non-drifted prediction — schema-driven payload using TRAINING_FEATURES.
# Uses representative values for the current dataset (Bank Marketing by default).
#
# NOTE: The payload uses values on the SAME SCALE as training data to avoid
# artificially inflating drift_magnitude in Section 5 / QuickSight. If you swap
# datasets, update _DATASET_DEFAULTS to match realistic feature ranges.
import json

# Build a sanity payload from the schema feature list with representative values.
# This dict works for Bank Marketing; if you switch datasets, update these values
# to match realistic feature ranges for YOUR schema.
_DATASET_DEFAULTS = {
    'age': 41, 'job': 0, 'marital': 1, 'education': 6, 'credit_default': 0,
    'housing': 2, 'loan': 0, 'contact': 1, 'month': 6, 'day_of_week': 1,
    'duration': 180, 'campaign': 2, 'pdays': 999, 'previous': 0,
    'poutcome': 1, 'emp_var_rate': 1.1, 'cons_price_idx': 93.994,
    'cons_conf_idx': -36.4, 'euribor3m': 4.857, 'nr_employed': 5191.0,
}
sanity_payload = {f: _DATASET_DEFAULTS.get(f, 0.0) for f in TRAINING_FEATURES}

result, latency_ms = invoke_endpoint_json(ENDPOINT_NAME, sanity_payload)
print(f"Prediction          : {result['prediction']}")
print(f"P({TARGET_COLUMN}=1): {result['probability_positive']:.4f}")
print(f"Latency             : {latency_ms:.0f} ms")
print(f"\nLogged to {ATHENA_DATABASE}.{ATHENA_INFERENCE_TABLE} (visible after SQS flush).")


### 3.3 Bulk drifted inferences

In [ ]:
# Bulk drifted inferences — requires Section 2 to have generated the drifted CSV.
NUM_BULK_TESTS = 101

if not CSV_DRIFTED_DATA.exists():
    print(f'⚠ Drifted CSV not found: {CSV_DRIFTED_DATA}')
    print('  Run Section 2 first to generate it, then re-run this cell.')
else:
    drifted_df = pd.read_csv(CSV_DRIFTED_DATA)
    for col in TRAINING_FEATURES:
        if col in drifted_df.columns:
            drifted_df[col] = pd.to_numeric(drifted_df[col], errors='coerce')
    drifted_df = drifted_df.dropna(subset=TRAINING_FEATURES, how='all')

    n = min(NUM_BULK_TESTS, len(drifted_df))
    samples = drifted_df.sample(n=n, random_state=123)

    print(f'Source           : {CSV_DRIFTED_DATA.name}')
    print(f'Sending          : {n} rows to {ENDPOINT_NAME}')
    print()

    predictions, latencies, errors = [], [], []
    for i, (_, row) in enumerate(samples.iterrows()):
        try:
            result, lat = invoke_endpoint_json(ENDPOINT_NAME, row[TRAINING_FEATURES].to_dict())
            predictions.append(result)
            latencies.append(lat)
        except Exception as e:
            errors.append(str(e))
        # Small delay so the endpoint's SQS batcher can flush each message
        if (i + 1) % 10 == 0:
            time.sleep(0.5)

    lat_arr = np.array(latencies) if latencies else np.array([0])
    positive = sum(1 for p in predictions if p['prediction'] == 1)
    print(f'Sent             : {len(predictions)}/{n}  ({len(errors)} errors)')
    print(f'Latency P50/P95/P99: {np.percentile(lat_arr, 50):.0f} / {np.percentile(lat_arr, 95):.0f} / {np.percentile(lat_arr, 99):.0f} ms')
    print(f'Positive ({TARGET_COLUMN}=1): {positive}/{len(predictions)} ({positive / max(len(predictions), 1) * 100:.1f}%)')
    print(f'\nWaiting 90s for SQS→Lambda→Athena flush...')
    time.sleep(90)
    print('✓ Flush wait complete.')

### 3.4 Verify predictions landed in Athena

In [ ]:
# Verify predictions landed in Athena.
query = f"""
SELECT COUNT(*) AS total,
       CAST(MIN(request_timestamp) AS TIMESTAMP(3)) AS earliest,
       CAST(MAX(request_timestamp) AS TIMESTAMP(3)) AS latest
FROM {ATHENA_DATABASE}.{ATHENA_INFERENCE_TABLE}
WHERE endpoint_name = '{ENDPOINT_NAME}'
"""
result = athena_client.execute_query(query)
if not result.empty and result['total'].iloc[0] > 0:
    print(f"✓ Found {result['total'].iloc[0]:,} predictions for {ENDPOINT_NAME}")
    print(f"  Time range: {result['earliest'].iloc[0]} → {result['latest'].iloc[0]}")
else:
    print('⚠ No predictions found yet — wait ~30s for the SQS→Lambda flush, then retry.')


## 4. Apply Ground Truth

In production, labels arrive asynchronously (e.g. from investigations, manual
review, or delayed outcome observation). For dev/test we simulate them here so
model-drift metrics have something to score.

**Do not re-run Section 3 after this point** — newly logged predictions would
have NULL ground_truth and wouldn't contribute to model drift until the
simulator runs again.

### 4.1 Simulator parameters

In [ ]:
# Ground truth simulation parameters from src/config/config.yaml → ground_truth_simulation.
# Only two knobs reduce simulated accuracy:
#   effective_accuracy = max(0.5, base_accuracy
#                                 - GROUND_TRUTH_SIM_FEATURE_DRIFT_IMPACT
#                                 - GROUND_TRUTH_SIM_MODEL_DRIFT_MAG)
print(f'Base accuracy        : {GROUND_TRUTH_SIM_ACCURACY:.2f}')
print(f'Feature-drift impact : -{GROUND_TRUTH_SIM_FEATURE_DRIFT_IMPACT:.2f}')
print(f'Model-drift magnitude: -{GROUND_TRUTH_SIM_MODEL_DRIFT_MAG:.2f}')
effective = max(0.5, GROUND_TRUTH_SIM_ACCURACY - GROUND_TRUTH_SIM_FEATURE_DRIFT_IMPACT - GROUND_TRUTH_SIM_MODEL_DRIFT_MAG)
print(f'\nEffective accuracy   : {effective:.2f}  → ~{int((1 - effective) * 100)}% of labels will be flipped')


### 4.2 Generate and apply simulated labels

In [ ]:
# The 90s wait in Cell 17 (after bulk inferences) already covers the SQS→Lambda
# flush cycle. This additional short wait ensures any stragglers from the sanity
# check or edge cases are flushed before we simulate ground truth.
import time
print('Waiting 30s for any remaining SQS→Lambda flush...')
time.sleep(30)

# Simulate ground truth programmatically (in notebook)
from src.drift_monitoring.simulate_ground_truth_from_athena import GroundTruthSimulator

# Create simulator with configured drift parameters
simulator = GroundTruthSimulator(
    athena_client=athena_client,
    endpoint_name=ENDPOINT_NAME,
    accuracy=GROUND_TRUTH_SIM_ACCURACY,
    positive_confirmation_days=(1, 7),
    negative_confirmation_days=(1, 30),
    feature_drift_impact=GROUND_TRUTH_SIM_FEATURE_DRIFT_IMPACT,
    model_drift_magnitude=GROUND_TRUTH_SIM_MODEL_DRIFT_MAG,
    seed=42
)

# Run simulation
print("Simulating ground truth for predictions without ground truth...")
stats = simulator.simulate_and_write(limit=None)

print("\n" + "=" * 80)
print("Simulation Summary")
print("=" * 80)

if stats['total_predictions'] > 0:
    print(f"Predictions processed: {stats['total_predictions']:,}")
    print(f"Ground truth updates created: {stats['updates_created']:,}")
    print(f"Actual positive: {stats.get('actual_fraud', stats.get('actual_positive', 0)):,}")
    print(f"False positives: {stats['false_positives']:,}")
    print(f"False negatives: {stats['false_negatives']:,}")
    print(f"Model accuracy: {stats.get('accuracy', GROUND_TRUTH_SIM_ACCURACY)*100:.1f}%")
    print("=" * 80)
    print("\n✅ Ground truth simulation complete!")
    print("   Next: Run the cell below to apply updates to inference_responses table")
else:
    print("⚠️  No predictions found to simulate ground truth")
    print("   Total predictions: 0")
    print("   Possible reasons:")
    print("   - No inference tests have been run yet (run Section 3)")
    print("   - All predictions already have ground truth")
    print("   - Wrong endpoint name filter")
    print("\n   Action: Run Section 3 to generate inference predictions first")
    print("=" * 80)

In [ ]:
# Apply simulated ground truth updates to inference_responses table
print("Applying ground truth updates to inference_responses table...")

# Initialize updater (already imported above)
if 'updater' not in dir():
    from src.drift_monitoring.update_ground_truth import GroundTruthUpdater
    updater = GroundTruthUpdater(athena_client=athena_client, dry_run=False)

# Get statistics before update
print("\nGetting update statistics...")
update_stats = updater.get_update_statistics()

if update_stats['total_updates'] > 0:
    print(f"\nPending updates: {update_stats['total_updates']:,}")
    print(f"  Positive (subscribed) cases: {update_stats['positive_cases']:,}")
    print(f"  False positives: {update_stats['false_positives']:,}")
    print(f"  False negatives: {update_stats['false_negatives']:,}")
    print(f"  Avg days to confirmation: {update_stats['avg_days_to_confirmation']:.1f}")
    
    # Execute the update
    print("\nMerging ground truth updates into inference_responses...")
    result = updater.update_ground_truth_batch()
    
    print(f"\n✅ Ground truth update complete!")
    print(f"   Records updated: {result['updated']:,}")
    print(f"\n   Next: Run the cell below to check coverage")
else:
    print("\n⚠️  No pending updates found")
    print("   Make sure you ran the simulation cell above first")

### 4.3 Verify coverage

In [ ]:
# Check current ground truth coverage before update
from src.drift_monitoring.update_ground_truth import GroundTruthUpdater

updater = GroundTruthUpdater(athena_client=athena_client, dry_run=False)

coverage_before = updater.get_coverage_statistics()
print('Current ground truth coverage:')
print(f"  Total predictions: {coverage_before['total_predictions']:,}")
print(f"  With ground truth: {coverage_before['with_ground_truth']:,} ({coverage_before['coverage_pct']:.2f}%)")
print(f"  Without ground truth: {coverage_before['without_ground_truth']:,}")

## 5. Detect Drift

The core of the notebook. Two independent checks, both traceable to the exact
model + data snapshot that produced the predictions (lineage pinned in
`baseline.json` on the registered ModelPackage):

- **5.2 Data drift** — Evidently `DataDriftPreset` (auto-selects KS / Chi-square
  for small samples, Wasserstein / Jensen-Shannon for n ≥ 1000). Baseline =
  `training_data`; current = `inference_responses`.
- **5.3 Model drift** — Evidently `ClassificationPreset` (ROC-AUC, precision,
  recall, F1). Baseline = `evaluation_data`; current = labeled
  `inference_responses`.

**One Evidently invocation, two consumers.** The Evidently output produced by
these cells is written to (a) the `monitoring_responses` Athena table and
(b) the MLflow experiment (metrics + Evidently HTML artifact). The scheduled
Lambda in Section 6 imports the *same* `run_data_drift_report()` and
`run_classification_report()` helpers used here (`src/drift_monitoring/evidently_reports.py`)
and writes its output the same way — so the numbers you see in this notebook,
in QuickSight, and in MLflow are always the same Evidently output. No custom
drift math is applied anywhere between compute and dashboard.

### 5.0 Run setup — `monitoring_run_id` + baseline lineage

Each pass produces one `monitoring_run_id`, reused by 5.2 / 5.3 / 5.4. This cell
also finds the last monitoring run so the drift checks scope the "current"
window to only predictions since then (re-running measures *new* drift, not all
cumulative traffic).

In [ ]:
# Resolve run id + lineage + the "since when" cutoff for the current window.
from src.drift_monitoring.baseline import load_baseline_from_registry
from datetime import datetime as _dt

monitoring_run_id = f'notebook-drift-{_dt.now().strftime("%Y%m%d-%H%M%S")}'
monitoring_run_ts = _dt.now()

# --- Baseline lineage (single lookup, reused by 6.2 / 6.3 / 6.4) ---
_baseline_meta = load_baseline_from_registry() or {}
model_package_arn      = _baseline_meta.get('model_package_arn') or '(not pinned)'
training_table         = _baseline_meta.get('training_table')   or ATHENA_TRAINING_TABLE
training_snapshot_id   = _baseline_meta.get('training_snapshot_id') or ''
evaluation_table       = _baseline_meta.get('evaluation_table') or ATHENA_EVALUATION_TABLE
evaluation_snapshot_id = _baseline_meta.get('evaluation_snapshot_id') or ''

# ROC-AUC the model scored on the held-out set at training time. Passed to
# ModelPerformanceMonitor in 5.1 so degradation is measured against training
# performance; without it the monitor falls back to the mean of the very windows
# it is judging, which can only ever report 0% degradation.
baseline_roc_auc = _baseline_meta.get('metrics', {}).get('roc_auc')

# --- Previous monitoring timestamp (delta filter for the "current" window) ---
prev_ts_query = f"""
    SELECT MAX(monitoring_timestamp) AS prev_ts
    FROM {ATHENA_DATABASE}.{MONITORING_TABLE_NAME}
    WHERE endpoint_name = '{ENDPOINT_NAME}'
"""
try:
    _prev = athena_client.execute_query(prev_ts_query)
    if _prev.empty:
        prev_monitoring_ts = None
    else:
        _val = _prev['prev_ts'].iloc[0]
        # pandas converts SQL NULL TIMESTAMP -> pd.NaT, which is NOT None
        # but ALSO can't be formatted as a TIMESTAMP literal. Treat both as cold-start.
        prev_monitoring_ts = None if pd.isna(_val) else _val
except Exception:
    # First-ever run: table empty / not yet readable. Cold-start fallback.
    prev_monitoring_ts = None

def _from(table, snapshot_id):
    return (f'{ATHENA_DATABASE}.{table} FOR VERSION AS OF {snapshot_id}'
            if snapshot_id else f'{ATHENA_DATABASE}.{table}')

def _snap_log(snapshot_id):
    return f'Iceberg snapshot {snapshot_id}' if snapshot_id else 'live table (no snapshot pinned)'

print('━' * 72)
print('MONITORING RUN — LINEAGE')
print('━' * 72)
print(f'  monitoring_run_id   : {monitoring_run_id}')
print(f'  Endpoint            : {ENDPOINT_NAME}')
print(f'  Model package ARN   : {model_package_arn}')
print(f'  Training baseline   : {ATHENA_DATABASE}.{training_table}  ({_snap_log(training_snapshot_id)})')
print(f'  Evaluation baseline : {ATHENA_DATABASE}.{evaluation_table}  ({_snap_log(evaluation_snapshot_id)})')
print(f'  Training ROC-AUC    : '
      + (f'{baseline_roc_auc:.4f}' if baseline_roc_auc else '(not recorded in baseline.json)'))
if prev_monitoring_ts is not None:
    print(f'  Current window      : predictions since last run @ {prev_monitoring_ts}')
else:
    print(f'  Current window      : cold-start fallback (last {MONITORING_DATA_DRIFT_LOOKBACK_DAYS} days for data drift, '
          f'last 30 days for model drift)')
print('━' * 72)


### 5.1 Overall classification metrics

`ModelPerformanceMonitor` computes precision/recall/F1/ROC-AUC against
`ground_truth` — the "current" half of the model-drift comparison, printed
inline before the detailed Evidently report.

In [ ]:
from src.drift_monitoring.monitor_model_performance import ModelPerformanceMonitor

monitor = ModelPerformanceMonitor(
    athena_client=athena_client,
    alert_threshold=0.85,  # ROC-AUC threshold for alerts
    min_samples=50,        # Minimum samples for reliable metrics
)

# Check ground truth coverage
coverage = monitor.get_ground_truth_coverage(endpoint_name=ENDPOINT_NAME, days=30)
print(f"Ground truth coverage (last 30 days):")
print(f"  Total predictions: {coverage['total_predictions']:,}")
print(f"  With ground truth: {coverage['with_ground_truth']:,} ({coverage['coverage_pct']:.2f}%)")

In [ ]:
# Generate performance report
report = monitor.generate_performance_report(
    endpoint_name=ENDPOINT_NAME,
    days=7,
    window='D',  # Daily time windows
    # Judge each window against training-time ROC-AUC (from baseline.json). With
    # baseline_roc_auc=None the monitor averages the windows it is judging, so
    # every alert reports "degradation: 0.0%" no matter how bad the window is.
    baseline_roc_auc=baseline_roc_auc,
)

monitor.print_report_summary(report)

In [ ]:
# Display overall metrics
if report.get('overall_metrics') and 'error' not in report['overall_metrics']:
    metrics = report['overall_metrics']
    print('Overall Model Performance:')
    print(f"  ROC-AUC:    {metrics.get('roc_auc', 'N/A')}")
    print(f"  PR-AUC:     {metrics.get('pr_auc', 'N/A')}")
    print(f"  Precision:  {metrics['precision']:.4f}")
    print(f"  Recall:     {metrics['recall']:.4f}")
    print(f"  F1-Score:   {metrics['f1_score']:.4f}")
    print(f"  Accuracy:   {metrics['accuracy']:.4f}")
    print(f"\nConfusion Matrix:")
    print(f"  TP: {metrics['true_positives']:,}  FP: {metrics['false_positives']:,}")
    print(f"  FN: {metrics['false_negatives']:,}  TN: {metrics['true_negatives']:,}")
else:
    print('Insufficient ground truth data for metrics. Run ground truth update first.')

### 5.2 Data drift

Per-feature drift test (Evidently `DataDriftPreset`). Baseline = `training_data`
(the distribution the model was trained on); current = `inference_responses`
within the window resolved in 5.0. Features that cannot be compared due to type
mismatches (e.g., string in training vs. encoded float in inference) are excluded.

**Which test runs?** Evidently auto-selects per column based on sample size:

| Column | Sample size | Test | Direction |
|---|---|---|---|
| Numeric | n < 1000 | Kolmogorov–Smirnov | p-value, lower = drift |
| Numeric | n ≥ 1000 | Wasserstein distance | distance, higher = drift |
| Categorical | n < 1000 | Chi-square | p-value, lower = drift |
| Categorical | n ≥ 1000 | Jensen-Shannon distance | distance, higher = drift |

Because raw `drift_score` has opposite drift directions across tests, the
notebook reports `drift_magnitude` (× past threshold) as the primary metric —
`1.0` = at threshold, `> 1.0` = drifted, higher = worse, regardless of test.

In [ ]:
import json as _json
from src.drift_monitoring.evidently_reports import run_data_drift_report

# ════════════════════════════════════════════════════════════════════════
# DATA DRIFT — feature-distribution shift (Evidently DataDriftPreset)
# ════════════════════════════════════════════════════════════════════════
# Baseline : `training_data` Iceberg slice (the distribution the model was
#            TRAINED on — this is the industry-standard reference for data
#            drift: did production traffic move away from what we taught it?).
# Current  : `inference_responses` rows since the last monitoring run
#            (or the last MONITORING_DATA_DRIFT_LOOKBACK_DAYS as cold-start).
# Method   : per-column test auto-selected by Evidently — KS/Chi-square
#            (p-value, lower = drift) for small samples, Wasserstein/
#            Jensen-Shannon (distance, higher = drift) for n ≥ 1000.
#            drift_magnitude ≥ 1.0 → drifted (test-agnostic).

BASELINE_LIMIT = 5000
INFERENCE_LIMIT = 10000
# All schema features are numeric (doubles) for the Bank Marketing dataset.
# If your dataset has non-numeric features that can't be KS-tested, exclude
# them here (e.g. string columns that weren't label-encoded).
NUMERIC_FEATURES = TRAINING_FEATURES

baseline_from = _from(training_table, training_snapshot_id)

# Current window: since last monitoring run if available, else cold-start lookback
if prev_monitoring_ts is not None:
    current_filter = f"request_timestamp > TIMESTAMP '{prev_monitoring_ts}'"
    window_label = f'since last monitoring run @ {prev_monitoring_ts}'
else:
    cold_start = (_dt.now() - timedelta(days=MONITORING_DATA_DRIFT_LOOKBACK_DAYS)).strftime('%Y-%m-%d %H:%M:%S')
    current_filter = f"request_timestamp >= TIMESTAMP '{cold_start}'"
    window_label = f'cold-start fallback (last {MONITORING_DATA_DRIFT_LOOKBACK_DAYS} days)'

print(f'Baseline source     : {ATHENA_DATABASE}.{training_table}  ({_snap_log(training_snapshot_id)})')
print(f'Current window      : {window_label}')
print(f'Features compared   : {len(NUMERIC_FEATURES)} numeric')

# --- Baseline rows ---
baseline_sql = f"""
    SELECT {", ".join(NUMERIC_FEATURES)}
    FROM {baseline_from}
    WHERE {TARGET_COLUMN} IS NOT NULL
    LIMIT {BASELINE_LIMIT}
"""
baseline_df = athena_client.execute_query(baseline_sql)
baseline_df = baseline_df.apply(pd.to_numeric, errors='coerce').dropna(how='all')
print(f'\nBaseline : {len(baseline_df):,} rows × {len(NUMERIC_FEATURES)} features')

# --- Current rows (parse JSON-encoded features) ---
inference_sql = f"""
    SELECT input_features
    FROM {ATHENA_DATABASE}.{ATHENA_INFERENCE_TABLE}
    WHERE endpoint_name = '{ENDPOINT_NAME}'
      AND {current_filter}
    LIMIT {INFERENCE_LIMIT}
"""
inf_raw = athena_client.execute_query(inference_sql)
print(f'Current  : {len(inf_raw):,} rows')

drift_results = None
if inf_raw.empty:
    print('\n⚠ No new inference data in the current window. Run Section 3 again, then re-run Section 5.')
elif len(inf_raw) < 50:
    print(f'\n⚠ Only {len(inf_raw)} rows — statistical tests need ≥ 50 samples for reliable results. Send more.')
else:
    current_df = pd.DataFrame([
        {f: float(d.get(f, float("nan"))) for f in NUMERIC_FEATURES}
        for d in (_json.loads(r) for r in inf_raw['input_features'])
    ]).dropna(how='all')

    valid = [c for c in NUMERIC_FEATURES
             if c in baseline_df.columns and c in current_df.columns
             and baseline_df[c].notna().any() and current_df[c].notna().any()]
    if len(valid) < len(NUMERIC_FEATURES):
        dropped = set(NUMERIC_FEATURES) - set(valid)
        print(f'  ⚠ Dropping {len(dropped)} all-NaN columns: {sorted(dropped)}')

    drift_results = run_data_drift_report(
        baseline_df=baseline_df[valid],
        current_df=current_df[valid],
    )

    print('\nRESULTS')
    print('━' * 72)
    # The dataset-level verdict is a SHARE rule, not "any column drifted": a run
    # can show several drifted features and still report False. Print the
    # threshold so the two numbers read as one decision.
    print(f'  Drift detected     : {drift_results["drift_detected"]}'
          f'  (dataset drifts when the drifted share reaches'
          f' {drift_results["drift_share_threshold"]:.0%})')
    print(f'  Drifted features   : {drift_results["drifted_columns_count"]} / {len(valid)}'
          f' ({drift_results["drifted_columns_share"]:.1%})')
    if drift_results["drift_detected"]:
        # Rank drifted columns by drift_magnitude (test-agnostic: 1.0 = at
        # threshold, higher = more drifted). Evidently picks KS/Chi-square
        # (p-value) or Wasserstein/Jensen-Shannon (distance) per column
        # based on sample size, so raw drift_score direction varies —
        # magnitude normalizes across tests.
        top_drifted = sorted(
            [(c, info) for c, info in drift_results.get('per_column', {}).items()
             if info.get('drifted', False)],
            key=lambda x: x[1].get('drift_magnitude', 0),
            reverse=True,
        )[:5]
        if top_drifted:
            print('  Top 5 drifted      :')
            for c, info in top_drifted:
                mag = info.get('drift_magnitude', 0)
                score = info.get('drift_score', 0)
                method = info.get('method', '?')
                print(f'    - {c:32s}  magnitude=×{mag:.1f}  (score={score:.4f}, method={method})')
        else:
            print('  Top 5 drifted      : (aggregate says drift, but no per-column matched — check evidently_reports.py parser)')
    print('━' * 72)

In [ ]:
# Display the interactive Evidently DataDriftPreset report.
if 'drift_results' in dir() and drift_results and drift_results.get('snapshot'):
    print('📊 Evidently Data Drift Report (interactive)')
    print('=' * 80)
    display(drift_results['snapshot'])
else:
    print('No drift results to visualize — re-run the cell above.')


In [ ]:
# Per-column drift summary, sorted by drift_magnitude descending (higher = more drifted).
# Evidently auto-selects the test per column, so raw drift_score direction
# varies (p-value: lower=drift, distance: higher=drift). drift_magnitude
# is test-agnostic: 1.0 = at threshold, > 1.0 = drifted.
if 'drift_results' in dir() and drift_results and drift_results.get('per_column'):
    print('📋 Per-Column Drift Summary')
    print('=' * 80)
    for c, info in sorted(
        drift_results['per_column'].items(),
        key=lambda x: x[1].get('drift_magnitude', 0),
        reverse=True,
    ):
        mag = info.get('drift_magnitude', 0)
        score = info.get('drift_score', 0)
        method = info.get('method', '?')
        status = '⚠ DRIFT' if info.get('drifted', False) else '✓ OK'
        print(f'    - {c:32s}  magnitude=×{mag:.1f}  (score={score:.4f}, method={method})  [{status}]')
else:
    print('No drift results — re-run the cell above.')

### 5.3 Model drift

Classification-metric drift (Evidently `ClassificationPreset`). Baseline
`(target, prediction)` pairs come from `evaluation_data` — re-scored here with
the registered artifact, because that table's own `prediction` column is NULL;
current `(ground_truth, prediction)` from labeled `inference_responses`.

**Where the degradation comes from.** Model drift is a *label* phenomenon, so it
needs ground truth — and in this lab the ground truth is simulated (Section 4).
The simulator sets `actual = prediction` and then flips `1 - effective_accuracy`
of the labels at random, so the measured ROC-AUC lands well below the training
baseline (~0.78 vs ~0.96 at the default 0.85 accuracy) and the monitor alerts.
Turning up `GROUND_TRUTH_SIM_FEATURE_DRIFT_IMPACT` / `GROUND_TRUTH_SIM_MODEL_DRIFT_MAG`
deepens it. Note this means the feature drift injected in Section 2 is **not**
what causes the degradation — data drift and model drift are simulated
independently here, which is also how the upstream reference sample does it.

If model drift is skipped, the cell prints why (too few labeled rows, or one
class only) — re-run Section 3.3 then Section 4.2.

In [ ]:
from src.drift_monitoring.evidently_reports import run_classification_report
from src.drift_monitoring.baseline import score_evaluation_baseline

# ════════════════════════════════════════════════════════════════════════
# MODEL DRIFT — Evidently ClassificationPreset
# ════════════════════════════════════════════════════════════════════════
# Baseline : (label, prediction) on `evaluation_data` — the held-out test set,
#            re-scored here with the registered model artifact. The table's own
#            `prediction` column is NULL (Lab 2 creates it, nothing backfills
#            it), so the baseline is computed rather than read.
# Current  : (ground_truth, prediction) from `inference_responses` since
#            the last monitoring run (or last 30 days cold-start).
#
# Both datasets must contain BOTH classes (0 and 1) in target AND prediction.

MIN_SAMPLES = 50
MODEL_DRIFT_LOOKBACK_DAYS = 30
PREDICTION_THRESHOLD = 0.5

# Current window: since last monitoring run, else cold-start
if prev_monitoring_ts is not None:
    days_window = MODEL_DRIFT_LOOKBACK_DAYS
    window_label = f'since last monitoring run @ {prev_monitoring_ts}'
else:
    days_window = MODEL_DRIFT_LOOKBACK_DAYS
    window_label = f'cold-start fallback (last {MODEL_DRIFT_LOOKBACK_DAYS} days)'

print(f'Baseline source     : {ATHENA_DATABASE}.{evaluation_table}  ({_snap_log(evaluation_snapshot_id)})')
print(f'Current window      : {window_label}')
print(f'Decision threshold  : probability_positive >= {PREDICTION_THRESHOLD}')

# Pull current predictions with ground truth, then apply the delta filter manually.
gt_all = monitor.load_predictions_with_ground_truth(
    endpoint_name=ENDPOINT_NAME, days=days_window,
)
gt_df = gt_all
if prev_monitoring_ts is not None and not gt_all.empty:
    gt_df = gt_all[gt_all['request_timestamp'] > prev_monitoring_ts]
print(f'\nCurrent  : {len(gt_df):,} predictions with ground truth')

# Use generic probability column (probability_positive is what the inference
# handler logs, regardless of dataset).
prob_col = 'probability_positive' if 'probability_positive' in gt_all.columns else 'probability_fraud'


def _model_drift_usable(df):
    """Can ClassificationPreset score this window? Returns (ok, reason).

    Evidently needs both classes in target AND prediction. On a short
    'since last monitoring run' window that is easy to miss — the positive
    class is ~11% of this dataset — and the old code then skipped model drift
    entirely, leaving a run that reports data drift only.
    """
    if len(df) < MIN_SAMPLES:
        return False, f'only {len(df)} labeled rows (need ≥ {MIN_SAMPLES})'
    if df['ground_truth'].nunique() < 2:
        return False, 'ground truth is single-class'
    if (df[prob_col] >= PREDICTION_THRESHOLD).nunique() < 2:
        return False, f'predictions are single-class at threshold {PREDICTION_THRESHOLD}'
    return True, ''


usable, why = _model_drift_usable(gt_df)
if not usable and len(gt_all) > len(gt_df):
    # Widen rather than skip: measuring model drift over the whole lookback is
    # still a real measurement, and it is what the Athena/QuickSight model-drift
    # columns are populated from (5.1 already uses a 7-day window, not the delta).
    print(f'⚠ Delta window unusable for model drift ({why}).')
    print(f'  Widening to the full {MODEL_DRIFT_LOOKBACK_DAYS}-day window ({len(gt_all):,} labeled rows).')
    gt_df = gt_all
    window_label = f'widened to last {MODEL_DRIFT_LOOKBACK_DAYS} days'
    usable, why = _model_drift_usable(gt_df)

model_report = None
if not usable:
    print(f'⚠ Skipping model drift — {why}.')
    print(f'  Fix: re-run Section 3.3 (more drifted inferences) then Section 4.2 (simulator),')
    print(f'  or lower PREDICTION_THRESHOLD below {PREDICTION_THRESHOLD}.')
else:
    current_df = pd.DataFrame({
        'target':     gt_df['ground_truth'].astype(int).values,
        'prediction': (gt_df[prob_col] >= PREDICTION_THRESHOLD).astype(int).values,
    })

    # Baseline: re-score the pinned evaluation snapshot with the artifact the
    # endpoint serves. Reproduces baseline.json's ROC-AUC exactly, which is the
    # check printed below.
    if not _baseline_meta.get('model_package_arn'):
        print('⚠ No baseline.json resolved in 5.0 — cannot score the held-out baseline.')
        baseline_df = None
    else:
        baseline_df = score_evaluation_baseline(
            baseline=_baseline_meta,
            athena_client=athena_client,
            feature_names=TRAINING_FEATURES,
            target_column=TARGET_COLUMN,
            threshold=PREDICTION_THRESHOLD,
        )
        print(f'Baseline : {len(baseline_df):,} predictions '
              f'({sum(baseline_df["target"] == 1)} positive / '
              f'{sum(baseline_df["target"] == 0)} negative labels)')

    # Evidently ClassificationPreset needs BOTH classes in BOTH target AND
    # prediction of BOTH datasets.
    if baseline_df is None:
        degenerate = ['baseline (no ModelPackage resolved in 5.0)']
    else:
        sides = [('baseline.target', baseline_df['target']),
                 ('baseline.prediction', baseline_df['prediction']),
                 ('current.target', current_df['target']),
                 ('current.prediction', current_df['prediction'])]
        degenerate = [name for name, col in sides if col.nunique() < 2]
    if degenerate:
        print(f'⚠ Skipping classification report — single-class column(s): {degenerate}')
        print(f'  Likely cause: model never predicted the minority class on this sample.')
        print(f'  Fix: send more drifted inferences, or lower PREDICTION_THRESHOLD below 0.5.')
    else:
        model_report = run_classification_report(
            # Only the two label columns — ClassificationPreset ignores the
            # probability column but Evidently still profiles every column given.
            baseline_df=baseline_df[['target', 'prediction']], current_df=current_df,
            target_column='target', prediction_column='prediction',
        )
        print('\n✓ Classification report ready — interactive view below.')

if model_report:
    print()
    print('Evidently Classification Report (interactive)')
    print('=' * 80)
    display(model_report['snapshot'])


### 5.4 Log results to MLflow and Athena

Two writes keyed on the same `monitoring_run_id`: (1) an MLflow run with all
drift metrics + the Evidently HTML reports as artifacts; (2) one row in
`monitoring_responses` (with model-package ARN + snapshot id) plus a backfill
tagging the inference rows this run measured — so QuickSight can join them.

In [ ]:
from src.drift_monitoring.log_monitoring_to_mlflow import log_monitoring_to_mlflow

drift_data   = drift_results if 'drift_results' in dir() else None
model_data   = model_report  if 'model_report'  in dir() else None
metrics_data = report.get('overall_metrics') if 'report' in dir() and report.get('overall_metrics') else None

result = log_monitoring_to_mlflow(
    drift_results=drift_data,
    model_report=model_data,
    overall_metrics=metrics_data,
    endpoint_name=ENDPOINT_NAME,
    model_name=MLFLOW_MODEL_NAME,
    region=REGION,
)
if result['success']:
    print(f"✓ Logged to MLflow — run_id={result['mlflow_run_id']}")
else:
    print(f"⚠ MLflow logging failed: {result.get('error', 'unknown')}")


In [ ]:
# Write the monitoring run + tag the inference rows it measured.
#
# Two Athena ops, both keyed on the SAME monitoring_run_id (resolved in 6.0):
#   1. INSERT one row into monitoring_responses (the drift verdict for this run)
#   2. UPDATE inference_responses to backfill monitoring_run_id on the
#      predictions that fell in this run's current window
#
# Column list in op #1 MUST match the monitoring_responses DDL in
# templates/4-inference-capture.yaml (the workshop stack's table-creator)
# AND the QuickSight dataset in src/governance/create_governance_dashboard.py.
import json as _json

# --- Aggregate drift findings into Athena-bound values ---
data_detected, drifted_count, drifted_share = False, 0, 0.0
features_analyzed, data_sample = 0, 0
# per_feature is a dict of {feature_name: {score, magnitude, method, threshold}}.
# We persist an object (not just the raw score) because Evidently auto-picks
# the test per column — raw drift_score has opposite drift directions
# depending on which test ran (p-value: lower=drift; distance: higher=drift).
# Downstream dashboards should compare `magnitude` (test-agnostic, ≥ 1.0 = drift).
per_feature = {}
if 'drift_results' in dir() and drift_results:
    data_detected     = drift_results.get('drift_detected', False)
    drifted_count     = drift_results.get('drifted_columns_count', 0)
    drifted_share     = drift_results.get('drifted_columns_share', 0)
    features_analyzed = drift_results.get('features_analyzed', 0)
    data_sample       = drift_results.get('sample_size', 0)
    for col, info in drift_results.get('per_column', {}).items():
        per_feature[col] = {
            'score':     info.get('drift_score', 0),
            'magnitude': info.get('drift_magnitude', 0),
            'method':    info.get('method', ''),
            'threshold': info.get('threshold', 0),
        }

mdrift = baseline_auc = current_auc = degrad = degrad_pct = None
acc = prec = rec = f1 = model_sample = None
if 'report' in dir() and report.get('overall_metrics') and 'error' not in report.get('overall_metrics', {}):
    om = report['overall_metrics']
    # ModelPerformanceMonitor reports the CURRENT window's metrics only; the
    # baseline and the degradation live in report['alerts'] (or are derived from
    # baseline.json here). Reading them off `om` leaves these columns NULL,
    # which is what the governance dashboard's model-drift panels read.
    current_auc  = om.get('roc_auc')
    baseline_auc = baseline_roc_auc
    if baseline_auc and current_auc is not None:
        degrad     = baseline_auc - current_auc
        degrad_pct = degrad / baseline_auc * 100
    # The monitor's own verdict: it raised an alert for at least one window.
    mdrift       = bool(report.get('alerts'))
    acc          = om.get('accuracy')
    prec         = om.get('precision')
    rec          = om.get('recall')
    if prec and rec and (prec + rec) > 0:
        f1 = 2 * prec * rec / (prec + rec)
    model_sample = om.get('sample_count')

def sql_val(v):
    if v is None:                return 'NULL'
    if isinstance(v, bool):      return 'TRUE' if v else 'FALSE'
    if isinstance(v, str):       return "'" + v.replace("'", "''") + "'"
    return str(v)

mp_arn = _baseline_meta.get('model_package_arn')
# Use the model-package short-id as the model_version label so it's readable
model_version_label = mp_arn.split('/')[-1] if mp_arn else 'latest'

columns = [
    'monitoring_run_id', 'monitoring_timestamp',
    'endpoint_name', 'model_version', 'model_package_arn',
    'evaluation_snapshot_id', 'training_snapshot_id',
    'data_drift_detected', 'drifted_columns_count', 'drifted_columns_share',
    'features_analyzed', 'data_sample_size', 'model_drift_detected',
    'baseline_roc_auc', 'current_roc_auc',
    'roc_auc_degradation', 'roc_auc_degradation_pct',
    'accuracy', 'precision', 'recall', 'f1_score',
    'model_sample_size', 'per_feature_drift_scores',
    'evidently_report_s3_path', 'mlflow_run_id',
    'alert_sent', 'detection_engine', 'created_at',
]
ts = f"TIMESTAMP '{monitoring_run_ts.strftime('%Y-%m-%d %H:%M:%S')}'"
values = [
    sql_val(monitoring_run_id),
    ts,
    sql_val(ENDPOINT_NAME),
    sql_val(model_version_label),
    sql_val(mp_arn),
    sql_val(evaluation_snapshot_id or None),
    sql_val(training_snapshot_id or None),
    sql_val(data_detected), sql_val(drifted_count), sql_val(drifted_share),
    sql_val(features_analyzed), sql_val(data_sample), sql_val(mdrift),
    sql_val(baseline_auc), sql_val(current_auc),
    sql_val(degrad), sql_val(degrad_pct),
    sql_val(acc), sql_val(prec), sql_val(rec), sql_val(f1),
    sql_val(model_sample),
    sql_val(_json.dumps(per_feature) if per_feature else None),
    sql_val(None),                                  # evidently_report_s3_path
    sql_val(result.get('mlflow_run_id') if 'result' in dir() and isinstance(result, dict) else None),
    sql_val(data_detected or mdrift),
    sql_val('evidently'),
    ts,
]
insert_sql = (
    f"INSERT INTO {ATHENA_DATABASE}.{MONITORING_TABLE_NAME} "
    f"({', '.join(columns)}) VALUES ({', '.join(values)})"
)

# --- 1) Write the monitoring_responses row ---
try:
    athena_client.execute_query(insert_sql, return_results=False)
    print(f'✓ Wrote {monitoring_run_id} to {ATHENA_DATABASE}.{MONITORING_TABLE_NAME}')
    print(f'  Model package ARN     : {mp_arn or "(not pinned)"}')
    print(f'  Evaluation snapshot id: {evaluation_snapshot_id or "(not pinned)"}')
    print(f'  Training snapshot id  : {training_snapshot_id or "(not pinned)"}')
    print(f'  Data drift detected   : {data_detected}')
    print(f'  Model drift detected  : {mdrift}')
except Exception as e:
    print(f'✗ Failed to write monitoring_responses row: {e}')

# --- 2) Backfill monitoring_run_id on the inference rows we just measured ---
# Window is the same one cells 6.2 / 6.3 used: (prev_monitoring_ts, monitoring_run_ts].
# UPDATE only touches rows still tagged NULL so re-runs of this cell are idempotent.
window_lower = (
    f"TIMESTAMP '{prev_monitoring_ts}'" if prev_monitoring_ts is not None
    else f"TIMESTAMP '{(_dt.now() - timedelta(days=MONITORING_DATA_DRIFT_LOOKBACK_DAYS)).strftime('%Y-%m-%d %H:%M:%S')}'"
)
window_upper = f"TIMESTAMP '{monitoring_run_ts.strftime('%Y-%m-%d %H:%M:%S')}'"

backfill_sql = f"""
    UPDATE {ATHENA_DATABASE}.{ATHENA_INFERENCE_TABLE}
    SET monitoring_run_id = '{monitoring_run_id}'
    WHERE endpoint_name = '{ENDPOINT_NAME}'
      AND monitoring_run_id IS NULL
      AND request_timestamp > {window_lower}
      AND request_timestamp <= {window_upper}
"""
try:
    athena_client.execute_query(backfill_sql, return_results=False)
    # Read back how many got tagged
    verify_sql = f"""
        SELECT COUNT(*) AS n FROM {ATHENA_DATABASE}.{ATHENA_INFERENCE_TABLE}
        WHERE monitoring_run_id = '{monitoring_run_id}'
    """
    res = athena_client.execute_query(verify_sql)
    n_tagged = int(res['n'].iloc[0]) if not res.empty else 0
    print(f'✓ Tagged {n_tagged} inference rows with monitoring_run_id={monitoring_run_id}')
except Exception as e:
    print(f'⚠ Could not backfill monitoring_run_id on inference_responses: {e}')
    print('  (Schema may not include the column yet — see CFN inference_responses DDL.)')


## Teardown

This notebook created no infrastructure, so there is nothing here to tear down —
the drifted CSV and the Evidently HTML reports are local files, and the drift
results are rows in the `bank_marketing` Athena tables that the workshop stack
owns.

Two things *are* still costing money after this lab, and neither belongs to this
notebook:

- **The endpoint.** Lab 3A's final cell deletes the endpoint, its config, and the
  model. Run it once you have finished every Lab 5 notebook — 5A through 5F all
  need a live endpoint.
- **The workshop stack.** When you are completely done:
  ```bash
  aws cloudformation delete-stack --stack-name bank-marketing-prediction-workshop
  ```
  The S3 buckets are retained on purpose; `scripts/delete_s3_buckets.py` empties
  and removes them once you no longer need the data.

---